In [1]:
#Cell 1 — load 2018 raw data + our saved feature selection report from Step 6:
import sys, os, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from src.utils import load_config, set_seed
from src.cross_dataset.alignment import (
    build_alignment_report, print_alignment_report, rename_2018_to_2017_schema
)

cfg = load_config()
set_seed(cfg["project"]["random_state"])

path_2018 = os.path.join(cfg["paths"]["cicids2018_dir"], "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv")
df_2018_raw = pd.read_csv(path_2018, low_memory=False)
df_2018_raw.columns = [str(c).strip() for c in df_2018_raw.columns]

print("2018 raw shape:", df_2018_raw.shape)
print("\nLabel distribution:")
print(df_2018_raw["Label"].value_counts())

with open(os.path.join(cfg["paths"]["reports_dir"], "selected_features.json")) as f:
    selection_info = json.load(f)

selected_features_2017 = selection_info["selected_features"]
print(f"\nLoaded {len(selected_features_2017)} selected features from Step 6.")

2018 raw shape: (1048575, 80)

Label distribution:
Label
DDOS attack-HOIC        686012
Benign                  360833
DDOS attack-LOIC-UDP      1730
Name: count, dtype: int64

Loaded 20 selected features from Step 6.


In [2]:
#Cell 2 — build and print the alignment report:
alignment_report = build_alignment_report(selected_features_2017)
print_alignment_report(alignment_report)

# Save the report for the thesis appendix
with open(os.path.join(cfg["paths"]["reports_dir"], "cross_dataset_alignment_report.json"), "w") as f:
    json.dump(alignment_report, f, indent=2)

CROSS-DATASET FEATURE ALIGNMENT REPORT
Total source (2017) features:     20
Successfully mapped to 2018:      20
Unmapped (would be dropped):      0
Final aligned feature count:      20
----------------------------------------------------------------
Mapping used (2017 -> 2018):
    Fwd Packet Length Max               -> Fwd Pkt Len Max
    Total Length of Fwd Packets         -> TotLen Fwd Pkts
    Destination Port                    -> Dst Port
    Init_Win_bytes_forward              -> Init Fwd Win Byts
    act_data_pkt_fwd                    -> Fwd Act Data Pkts
    Bwd Packet Length Max               -> Bwd Pkt Len Max
    Fwd Packet Length Mean              -> Fwd Pkt Len Mean
    Total Fwd Packets                   -> Tot Fwd Pkts
    Bwd Packet Length Min               -> Bwd Pkt Len Min
    Init_Win_bytes_backward             -> Init Bwd Win Byts
    Fwd IAT Mean                        -> Fwd IAT Mean
    Flow IAT Std                        -> Flow IAT Std
    Bwd IAT Total    

In [3]:
#Cell 3 — apply alignment (rename 2018 columns to match 2017 schema):
X_2018_aligned = rename_2018_to_2017_schema(df_2018_raw, alignment_report)

print("Aligned 2018 feature matrix shape:", X_2018_aligned.shape)
print("\nColumn names (should exactly match your 2017 selected features):")
print(list(X_2018_aligned.columns))

print("\nFirst 3 rows:")
print(X_2018_aligned.head(3))

Aligned 2018 feature matrix shape: (1048575, 20)

Column names (should exactly match your 2017 selected features):
['Fwd Packet Length Max', 'Total Length of Fwd Packets', 'Destination Port', 'Init_Win_bytes_forward', 'act_data_pkt_fwd', 'Bwd Packet Length Max', 'Fwd Packet Length Mean', 'Total Fwd Packets', 'Bwd Packet Length Min', 'Init_Win_bytes_backward', 'Fwd IAT Mean', 'Flow IAT Std', 'Bwd IAT Total', 'Bwd Packets/s', 'Max Packet Length', 'Packet Length Mean', 'Bwd IAT Mean', 'URG Flag Count', 'Packet Length Variance', 'Fwd Packet Length Min']

First 3 rows:
   Fwd Packet Length Max  Total Length of Fwd Packets  Destination Port  \
0                    135                          135                80   
1                    500                         1500               500   
2                    500                         1500               500   

   Init_Win_bytes_forward  act_data_pkt_fwd  Bwd Packet Length Max  \
0                   29200                 1               